# 🔭 Intelligent Astronomical Pattern Analysis Using Decision Trees

**Goal:** Classify astronomical objects into **STAR**, **GALAXY** and **QSO (quasar)** using their photometric colours and redshift, and discover which patterns a Decision Tree uses to separate them.

**Workflow:** Data → EDA → Feature Engineering → Decision Tree → Tuning → Evaluation → Pattern Interpretation → Prediction

> By default the notebook generates an SDSS-style dataset so it runs anywhere with zero setup. To use the real SDSS DR16 CSV (Kaggle), set `USE_REAL_DATA = True` in Step 2.

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_style("whitegrid")
np.random.seed(42)
print("Libraries loaded ✅")

## Step 2: Load the dataset

Features follow the SDSS photometric system:
- `u, g, r, i, z` – brightness (magnitudes) in five wavelength bands (ultraviolet → infrared)
- `redshift` – how much the light is stretched (proxy for distance/velocity)
- `class` – target label (STAR / GALAXY / QSO)

In [ ]:
USE_REAL_DATA = False   # set True to upload the real SDSS CSV (needs columns u,g,r,i,z,redshift,class)

def generate_sdss_like(n=6000, seed=42):
    rng = np.random.default_rng(seed)
    n_gal, n_star, n_qso = int(n*0.5), int(n*0.35), n - int(n*0.5) - int(n*0.35)

    def make(n, r_mean, r_sd, colors, color_sd, z_func, label):
        r = rng.normal(r_mean, r_sd, n)
        ug = rng.normal(colors[0], color_sd[0], n)
        gr = rng.normal(colors[1], color_sd[1], n)
        ri = rng.normal(colors[2], color_sd[2], n)
        iz = rng.normal(colors[3], color_sd[3], n)
        i_ = r - ri; g_ = r + gr; u_ = g_ + ug; z_ = i_ - iz
        return pd.DataFrame({"u":u_, "g":g_, "r":r, "i":i_, "z":z_,
                             "redshift": z_func(n), "class": label})

    gal  = make(n_gal, 17.8, 1.2, (2.0, 0.85, 0.40, 0.30), (0.45, 0.30, 0.15, 0.12),
                lambda k: rng.gamma(2.5, 0.07, k), "GALAXY")
    star = make(n_star, 17.0, 1.5, (1.7, 0.75, 0.30, 0.20), (0.80, 0.50, 0.35, 0.25),
                lambda k: np.abs(rng.normal(0, 0.0007, k)), "STAR")
    qso  = make(n_qso, 18.6, 0.9, (0.45, 0.25, 0.20, 0.15), (0.30, 0.20, 0.15, 0.12),
                lambda k: rng.gamma(4.0, 0.45, k), "QSO")
    return pd.concat([gal, star, qso]).sample(frac=1, random_state=seed).reset_index(drop=True)

if USE_REAL_DATA:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])
    df = df[["u","g","r","i","z","redshift","class"]]
else:
    df = generate_sdss_like()

print("Shape:", df.shape)
df.head()

## Step 3: Explore the data (EDA)

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. Class balance
df["class"].value_counts().plot(kind="bar", ax=ax[0], color=["#4c72b0","#dd8452","#55a868"])
ax[0].set_title("Class distribution"); ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=0)

# 2. Redshift by class
for c in df["class"].unique():
    ax[1].hist(df.loc[df["class"]==c, "redshift"].clip(upper=4), bins=50, alpha=0.6, label=c)
ax[1].set_title("Redshift distribution"); ax[1].set_xlabel("redshift"); ax[1].legend()

# 3. Colour-colour diagram
for c in df["class"].unique():
    s = df[df["class"]==c]
    ax[2].scatter(s["u"]-s["g"], s["g"]-s["r"], s=5, alpha=0.4, label=c)
ax[2].set_xlabel("u - g"); ax[2].set_ylabel("g - r"); ax[2].set_title("Colour-colour diagram"); ax[2].legend(markerscale=4)
ax[2].set_xlim(-1, 4); ax[2].set_ylim(-1, 3)
plt.tight_layout(); plt.show()

## Step 4: Feature engineering

Astronomers rarely use raw brightness. **Colour indices** (difference between two bands) reveal an object's temperature and physics, and are distance-independent. We create `u-g`, `g-r`, `r-i`, `i-z`.

In [ ]:
df["u_g"] = df["u"] - df["g"]
df["g_r"] = df["g"] - df["r"]
df["r_i"] = df["r"] - df["i"]
df["i_z"] = df["i"] - df["z"]

features = ["u","g","r","i","z","redshift","u_g","g_r","r_i","i_z"]
X = df[features]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

## Step 5: Baseline Decision Tree

In [ ]:
base = DecisionTreeClassifier(random_state=42)
base.fit(X_train, y_train)
print("Baseline (unrestricted tree)")
print("  Train accuracy:", round(base.score(X_train, y_train), 4))
print("  Test  accuracy:", round(base.score(X_test,  y_test), 4))
print("  Tree depth    :", base.get_depth(), "| Leaves:", base.get_n_leaves())

An unrestricted tree usually **memorises** the training data (train ≈ 100%) and is deeper than needed. Let's see how depth affects performance.

In [ ]:
depths = range(1, 21)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    tr.append(m.score(X_train, y_train)); te.append(m.score(X_test, y_test))

plt.figure(figsize=(8,4.5))
plt.plot(depths, tr, "o-", label="Train")
plt.plot(depths, te, "s-", label="Test")
plt.xlabel("max_depth"); plt.ylabel("Accuracy"); plt.title("Over-fitting analysis")
plt.legend(); plt.xticks(list(depths)); plt.show()

## Step 6: Hyperparameter tuning (GridSearchCV)

In [ ]:
param_grid = {
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 10, 30],
    "min_samples_leaf": [1, 5, 15],
    "criterion": ["gini", "entropy"],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,
                    cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters :", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))
model = grid.best_estimator_

## Step 7: Evaluate the final model

In [ ]:
y_pred = model.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
cv = cross_val_score(model, X, y, cv=5)
print("5-fold CV    : %.4f ± %.4f\n" % (cv.mean(), cv.std()))
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(5.5,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=model.classes_),
                       display_labels=model.classes_).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix"); plt.show()

## Step 8: Pattern discovery – what did the tree learn?
Feature importance shows which measurements matter most. The tree diagram and text rules reveal the actual decision patterns.

In [ ]:
imp = pd.Series(model.feature_importances_, index=features).sort_values()
imp.plot(kind="barh", figsize=(7,4.5), color="#4c72b0")
plt.title("Feature importance"); plt.xlabel("Importance"); plt.show()

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(model, max_depth=3, feature_names=features, class_names=model.classes_,
          filled=True, rounded=True, fontsize=9)
plt.title("Decision Tree (top 3 levels)"); plt.show()

In [ ]:
print(export_text(model, feature_names=features, max_depth=3))

## Step 9: Visualise the decision boundary on the two strongest features

In [ ]:
top2 = list(imp.sort_values(ascending=False).index[:2])
print("Top 2 features:", top2)

m2 = DecisionTreeClassifier(max_depth=model.get_depth(), random_state=42).fit(X_train[top2], y_train)
print("Accuracy using only these 2 features:", round(m2.score(X_test[top2], y_test), 4))

x_min, x_max = X[top2[0]].quantile([0.01, 0.99])
y_min, y_max = X[top2[1]].quantile([0.01, 0.99])
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = m2.predict(pd.DataFrame({top2[0]: xx.ravel(), top2[1]: yy.ravel()}))
Z = pd.Categorical(Z, categories=m2.classes_).codes.reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.25, levels=np.arange(-0.5, len(m2.classes_)+0.5), cmap="viridis")
for c in m2.classes_:
    s = df[df["class"]==c].sample(400, random_state=1)
    plt.scatter(s[top2[0]], s[top2[1]], s=10, label=c)
plt.xlim(x_min, x_max); plt.ylim(y_min, y_max)
plt.xlabel(top2[0]); plt.ylabel(top2[1]); plt.title("Decision regions"); plt.legend(); plt.show()

## Step 10: Classify a new astronomical object

In [ ]:
def classify(u, g, r, i, z, redshift):
    row = pd.DataFrame([{"u":u,"g":g,"r":r,"i":i,"z":z,"redshift":redshift,
                         "u_g":u-g,"g_r":g-r,"r_i":r-i,"i_z":i-z}])[features]
    pred = model.predict(row)[0]
    probs = {c: round(float(p), 3) for c, p in zip(model.classes_, model.predict_proba(row)[0])}
    return pred, probs

samples = {
    "Nearby faint object":     (19.5, 17.9, 17.1, 16.7, 16.4, 0.09),
    "Very distant, blue":      (19.2, 18.8, 18.6, 18.4, 18.3, 2.10),
    "Zero-redshift, bright":   (16.2, 15.0, 14.6, 14.4, 14.3, 0.0002),
}
for name, vals in samples.items():
    p, pr = classify(*vals)
    print(f"{name:25s} -> {p:7s} {pr}")

## Step 11: Conclusion

- A **Decision Tree** separates stars, galaxies and quasars with high accuracy using colours and redshift.
- **Redshift** and **colour indices** (e.g. `u-g`) are the most informative features: stars sit at ~0 redshift, galaxies at low-moderate redshift with redder colours, quasars at high redshift with bluer colours.
- **Depth limiting / tuning** reduces over-fitting and keeps the model interpretable. Rules can be read directly from the tree.
- **Future work:** Random Forest / Gradient Boosting, real SDSS DR17 data, handling class imbalance, and unsupervised pattern discovery (clustering) for unknown object types.